# ZrN/VN Oxidation — matching Ji et al. J. Phys. Chem. C 2025 protocol

Protocol (based on the paper; deviations noted):
- Add 6 O atoms (ZrN) or 10 O atoms (VN) above surface each round
- Run NVT MD for 20 ps per round (8000 steps at 2.5 fs timestep; Ji et al. use 6.25 ps)
- Remove all unbound species: free O, O2, NO, N2 — anything that did not bind to surface
- Repeat until saturation (added O no longer incorporates)
- Thermostat: Langevin, friction 0.001 fs^-1 (Ji et al. use a Nose-Hoover chain)
- Temperatures: 295, 363, 873, 1023 K

## Imports

In [ ]:
import numpy as np
from ase import io, units, Atoms
from ase.geometry import get_distances, find_mic
from ase.neighborlist import build_neighbor_list, natural_cutoffs
from ase.data import covalent_radii
from ase.constraints import FixAtoms
from ase.md.nose_hoover_chain import NoseHooverChainNVT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.langevin import Langevin
import os
import shutil
import pfp_api_client
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator, EstimatorCalcMode

# Use PBE to match Ji's VASP PBE functional
estimator = Estimator(calc_mode=EstimatorCalcMode.PBE, model_version='v9.0.0')
calculator = ASECalculator(estimator)

print(f'pfp_api_client: {pfp_api_client.__version__}')

## Parameters — edit these for each run

In [ ]:
# Path to BASE directory (seed subfolder created automatically)
base_directory = '/home/jovyan/Matlantis/ZrN_100/363K'

# Random seed — change to 1,2,3,4,5 for each repeat run
# Controls both MB velocity initialisation and O placement randomness
random_seed = 1

# Temperature 
temperature = 363   # K

# MD parameters
timestep      = 2.5    # fs
tdamp         = 250.0  # fs 
md_steps      = 8000   # steps x 2.5 fs = 20 ps per round
log_steps     = 40

# Number of O atoms to add per round
# 6 for ZrN(100) and ZrN(110), 10 for VN(111)
n_o_per_round = 6

# Maximum number of rounds
max_rounds = 40

# Height above surface to place O atoms (Å above topmost slab atom)
o_height_above_surface = 1.5

# ── Derived: seed subfolder ──────────────────────────────────────────────────
directory = f'{base_directory}/seed_{random_seed}'
os.makedirs(directory, exist_ok=True)
print(f'Output directory: {directory}')
print(f'Random seed: {random_seed}')


## Helper functions

In [ ]:
def get_surface_top(atoms):
    """Get z-coordinate of topmost non-O atom (top of nitride slab)."""
    syms = atoms.get_chemical_symbols()
    pos = atoms.get_positions()
    # Top of the nitride slab = highest Zr or V or N atom
    slab_idx = [i for i,s in enumerate(syms) if s in ('Zr', 'V', 'N')]
    return pos[slab_idx, 2].max()


def add_o_atoms(atoms, n_atoms, rng, height_above):
    """
    Add n_atoms oxygen atoms at random x,y positions above the surface,
    matching Ji's approach of placing O atoms above the nitride surface.
    Each O atom gets a downward initial velocity (~750 m/s) with small
    angular spread, consistent with the Etch-run group script convention.
    """
    surface_top = get_surface_top(atoms)
    z_place = surface_top + height_above
    
    print(f'  Surface top at z={surface_top:.2f} Å, placing O at z={z_place:.2f} Å')
    
    added = 0
    attempts = 0
    
    while added < n_atoms:
        attempts += 1
        if attempts > 1000:
            print(f'  WARNING: could not place all O atoms after 1000 attempts, placed {added}/{n_atoms}')
            break
        
        # Random x,y within cell
        frac_x, frac_y = rng.random(2)
        pos_xy = atoms.cell.cartesian_positions([frac_x, frac_y, 0.0])[:2]
        new_pos = np.array([pos_xy[0], pos_xy[1], z_place])
        
        # Check minimum distance to existing atoms
        dists = np.linalg.norm(atoms.get_positions() - new_pos, axis=1)
        if dists.min() < 1.5:
            continue  # too close, try again
        
        vel = np.array([rng.uniform(-0.1, 0.1), rng.uniform(-0.1, 0.1), -750.0]) * (units.m / units.s)
        
        # Create O atom and append
        o_atom = Atoms('O', positions=[new_pos], momenta=[vel])
        atoms += o_atom
        added += 1
    
    print(f'  Added {added} O atoms above surface')
    return atoms


def find_molecules(atoms, cutoff_scale=1.2):
    """
    Identify connected molecules using neighbour list.
    Returns list of sets, each set containing atom indices of one molecule.
    """
    cutoffs = [covalent_radii[z] * cutoff_scale for z in atoms.numbers]
    
    from ase.neighborlist import PrimitiveNeighborList
    nl = PrimitiveNeighborList(
        cutoffs, skin=0.0, self_interaction=False, bothways=True
    )
    nl.update(atoms.pbc, atoms.get_cell(complete=True), atoms.get_positions())
    
    n = len(atoms)
    visited = np.zeros(n, dtype=bool)
    molecules = []
    
    for start in range(n):
        if visited[start]:
            continue
        # BFS
        mol = set()
        queue = [start]
        while queue:
            idx = queue.pop()
            if idx in mol:
                continue
            mol.add(idx)
            visited[idx] = True
            neighbours, _ = nl.get_neighbors(idx)
            for n_idx in neighbours:
                if n_idx not in mol:
                    queue.append(n_idx)
        molecules.append(mol)
    
    return molecules


def remove_unbound_species(atoms, surface_top_z, bind_threshold=3.5):
    """
    Remove species that did not bind to the surface, following Ji's protocol:
    - free O atoms that are not bonded to surface atoms
    - O2 molecules (recombined oxygen)
    - NO molecules
    - N2 molecules
    
    A species is considered 'bound' if it contains a Zr or V atom, OR
    if it is an O/N atom bonded to a Zr/V/N atom within the slab.
    
    Gas-phase species above the surface are identified as molecules
    that contain ONLY O and/or N atoms and have no bond to Zr/V.
    """
    syms = np.array(atoms.get_chemical_symbols())
    pos = atoms.get_positions()
    
    molecules = find_molecules(atoms)
    
    remove_indices = set()
    
    for mol in molecules:
        mol_syms = set(syms[list(mol)])
        mol_list = list(mol)
        
        # Skip if molecule contains Zr or V — it's part of the slab
        if 'Zr' in mol_syms or 'V' in mol_syms:
            continue
        
        # This molecule contains only O and/or N
        formula = ''.join(sorted(syms[mol_list]))
        mol_z = pos[mol_list, 2].mean()
        
        # Check if it's actually in gas phase (above slab)
        # or loosely sitting above surface with no Zr/V bond
        if set(mol_syms).issubset({'O', 'N'}):
            # Check for any bond to a Zr or V atom in the slab
            metal_idx = [i for i,s in enumerate(syms) if s in ('Zr', 'V')]
            metal_pos = pos[metal_idx]
            
            min_dist_to_metal = min(
                np.linalg.norm(pos[j] - mp)
                for j in mol_list
                for mp in metal_pos
            ) if metal_pos.size > 0 else 999
            
            if min_dist_to_metal > bind_threshold:
                # Not bound to any metal — this is a gas-phase species
                print(f'  Removing unbound species: {formula} '
                      f'(z={mol_z:.2f} Å, min dist to metal={min_dist_to_metal:.2f} Å)')
                remove_indices.update(mol_list)
    
    keep = [i for i in range(len(atoms)) if i not in remove_indices]
    atoms_clean = atoms[keep]
    
    n_removed = len(atoms) - len(atoms_clean)
    if n_removed > 0:
        print(f'  Removed {n_removed} unbound atoms total')
    else:
        print(f'  No unbound species found')
    
    return atoms_clean


def count_bound_o(atoms):
    """Count O atoms that are bound to the surface (bonded to Zr or V)."""
    syms = atoms.get_chemical_symbols()
    pos = atoms.get_positions()
    
    o_idx = [i for i,s in enumerate(syms) if s=='O']
    metal_idx = [i for i,s in enumerate(syms) if s in ('Zr', 'V')]
    
    if not o_idx or not metal_idx:
        return 0
    
    bound = 0
    for oi in o_idx:
        dists = np.linalg.norm(pos[metal_idx] - pos[oi], axis=1)
        if dists.min() < 3.5:
            bound += 1
    return bound


def print_composition(atoms, label=''):
    """Print current slab composition."""
    syms = atoms.get_chemical_symbols()
    counts = {s: syms.count(s) for s in sorted(set(syms))}
    bound_o = count_bound_o(atoms)
    print(f'  {label}Composition: {counts}, bound O: {bound_o}')

## Load starting structure

Start from the clean relaxed slab (no O atoms). If your surface_relaxed.traj has O atoms from the starting OUTCAR, they are stripped here to match Ji's clean starting surface.

In [ ]:
rng = np.random.default_rng(random_seed)

atoms = io.read(f'{directory}/surface_relaxed.traj', index=-1)
atoms.set_pbc([True, True, False])

# Strip any O atoms from starting structure to get clean bare slab
syms = atoms.get_chemical_symbols()
o_in_start = syms.count('O')
if o_in_start > 0:
    print(f'Stripping {o_in_start} O atoms from starting structure to get clean slab')
    keep = [i for i,s in enumerate(syms) if s != 'O']
    atoms = atoms[keep]

# Check for unphysically close atoms (e.g. O2 molecules) which cause MD explosions
from itertools import combinations as _comb
_pos = atoms.get_positions()
_min_d = min(np.linalg.norm(_pos[i]-_pos[j]) for i,j in _comb(range(len(atoms)), 2))
print(f'Min interatomic distance in starting structure: {_min_d:.3f} Å')
if _min_d < 1.5:
    print(f'WARNING: atoms too close ({_min_d:.3f} Å) — pre-relaxing starting structure...')
    from ase.optimize import FIRE as _FIRE
    atoms.calc = calculator
    _p = atoms.get_positions()
    _zlo, _zhi = _p[:,2].min(), _p[:,2].max()
    _fix = [i for i,p in enumerate(_p) if p[2] < _zlo + (_zhi-_zlo)*0.2]
    atoms.set_constraint(FixAtoms(indices=_fix))
    _FIRE(atoms, logfile=f'{directory}/surface_prerelax.log').run(fmax=0.05)
    atoms.set_constraint()  # clear — MD loop sets its own
    _p2 = atoms.get_positions()
    _min_d2 = min(np.linalg.norm(_p2[i]-_p2[j]) for i,j in _comb(range(len(atoms)), 2))
    print(f'Min distance after pre-relaxation: {_min_d2:.3f} Å')
    io.write(f'{directory}/surface_relaxed.traj', atoms)
    print('Saved pre-relaxed structure to surface_relaxed.traj')
else:
    print('Starting structure looks clean — no pre-relaxation needed')

print(f'Starting formula: {atoms.get_chemical_formula()}')
print(f'Number of atoms: {len(atoms)}')
print(f'Cell: {atoms.get_cell()}')
print(f'PBC: {atoms.get_pbc()}')

surface_top = get_surface_top(atoms)
print(f'Surface top at z={surface_top:.2f} Å')

## Run iterative oxidation — Ji protocol

Each round:
1. Add n_o_per_round O atoms above surface
2. Run NVT MD for 20 ps
3. Remove unbound species (free O, O2, NO, N2)
4. Check for saturation
5. Save structure

In [ ]:
log_file = f'{directory}/MD.log'
bound_o_history = []

# Delete old log file so this run has a clean log with no previous runs mixed in
if os.path.exists(log_file):
    os.remove(log_file)
    print(f'Deleted old log file: {log_file}')


# Save seed info for reproducibility tracking
import json as _json, datetime
_seed_info = {
    'seed': random_seed,
    'temperature': temperature,
    'base_directory': base_directory,
    'md_steps': md_steps,
    'n_o_per_round': n_o_per_round,
    'max_rounds': max_rounds,
    'timestamp': datetime.datetime.now().isoformat(),
}
with open(f'{directory}/seed_info.json', 'w') as _f:
    _json.dump(_seed_info, _f, indent=2)
print(f'Seed info saved to {directory}/seed_info.json')

# Initialise velocities from Maxwell-Boltzmann distribution at target temperature
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature)
Stationary(atoms)  # remove centre-of-mass drift
print(f'Initialised MB velocities at {temperature} K')

# Fix bottom layer of slab (bottom 20% of slab z-range) to prevent slab drift
pos_init = atoms.get_positions()
slab_z_min = pos_init[:, 2].min()
slab_z_max = get_surface_top(atoms)
slab_z_range = slab_z_max - slab_z_min
bottom_cutoff = slab_z_min + slab_z_range * 0.2
fixed_idx = [i for i, p in enumerate(pos_init) if p[2] < bottom_cutoff]
print(f'Fixing {len(fixed_idx)} bottom-layer atoms (z < {bottom_cutoff:.2f} Å)')

for round_i in range(max_rounds):
    print(f'\n{"="*60}')
    print(f'ROUND {round_i + 1} of {max_rounds}')
    print(f'{"="*60}')
    
    print_composition(atoms, label='Before addition: ')
    n_bound_before = count_bound_o(atoms)
    
    # --- Step 1: Add O atoms ---
    print(f'Adding {n_o_per_round} O atoms...')
    atoms = add_o_atoms(atoms, n_o_per_round, rng, o_height_above_surface)
    
    # --- Step 2: Run NVT MD ---
    print(f'Running NVT MD at {temperature} K for {md_steps * timestep / 1000:.3f} ps...')
    
    output_file     = f'{directory}/{round_i:0>2d}_MD.traj'
    trajectory_file = f'{directory}/{round_i:0>2d}_MD-trajectory.traj'
    
    atoms.calc = calculator
    
    # Fix bottom layer
    current_fixed = [i for i in fixed_idx if i < len(atoms)]
    atoms.set_constraint(FixAtoms(indices=current_fixed))
    
    dyn = Langevin(
    atoms,
    timestep=timestep * units.fs,
    temperature_K=temperature,
    friction=0.001 / units.fs,  # friction coefficient
    logfile=log_file,
    trajectory=trajectory_file,
    loginterval=log_steps
    )
    status = dyn.run(md_steps)
    print(f'  MD complete, status: {status}')
    
    # --- Step 3: Remove unbound species (Ji protocol) ---
    print('Removing unbound species (free O, O2, NO, N2)...')
    surface_top_now = get_surface_top(atoms)
    atoms = remove_unbound_species(atoms, surface_top_z=surface_top_now)
    min_atoms = 60 if 'ZrN' in directory else 100
    if len(atoms) < min_atoms:
        print(f"WARNING: slab has lost too many atoms ({len(atoms)}), stopping")
        break
        
    # Also hard-remove anything above 60 Å (stranded atoms)
    extreme = atoms.get_positions()[:, 2] > 60.0
    if extreme.any():
        print(f'  Hard-removing {extreme.sum()} atoms above 60 Å')
        atoms = atoms[~extreme]
    
    # --- Step 4: Check saturation ---
    n_bound_after = count_bound_o(atoms)
    bound_o_history.append(n_bound_after)
    
    print_composition(atoms, label='After cleanup: ')
    print(f'  Bound O this round: {n_bound_before} -> {n_bound_after} '
          f'(+{n_bound_after - n_bound_before})')
    
    # Save structure after this round
    io.write(output_file, atoms)
    print(f'  Saved to {output_file}')
    # No early stopping — run all rounds and judge saturation from bound_o_history

print(f'\n{"="*60}')
print('COMPLETE')
print(f'Bound O history: {bound_o_history}')
print(f'Final formula: {atoms.get_chemical_formula()}')
print(f'Final bound O count: {count_bound_o(atoms)}')
print(f'\nCompare to Ji Table 1:')
print(f'  ZrN(100) 295K: Ji got ZrO8/ZrN  (8 bound O)')
print(f'  ZrN(100) 363K: Ji got ZrO12/ZrN (12 bound O)')
print(f'  ZrN(110) 295K: Ji got ZrO18/ZrN (18 bound O)')
print(f'  ZrN(110) 363K: Ji got ZrO20/ZrN (20 bound O)')
print(f'  VN(111)  295K: Ji got VO11/VN   (11 bound O)')
print(f'  VN(111)  363K: Ji got VO18/VN   (18 bound O)')
print(f'  VN(111)  873K: Ji got VO12/VN   (12 bound O)')

## Final relaxation

Ji relaxes the final saturated structure. Run this cell after the loop completes.

In [ ]:
from ase.optimize import FIRE

print('Running final geometry relaxation...')
atoms.calc = calculator
# atoms.set_constraint(FixAtoms(indices=[i for i in fixed_idx if i < len(atoms)]))

dyn = FIRE(atoms, logfile=f'{directory}/relax_final.log')
dyn.run(fmax=0.05)

io.write(f'{directory}/final_relaxed.traj', atoms)
print(f'Final relaxed structure saved to {directory}/final_relaxed.traj')
print(f'Final formula: {atoms.get_chemical_formula()}')
print(f'Final bound O: {count_bound_o(atoms)}')